# Meta-Learning: Prototypical Networks, MAML, and Reptile

This notebook explores metric-based and gradient-based meta-learning algorithms:
1. **Prototypical Networks**: Few-shot classification via centroid prototype representations under Bregman divergences.
2. **MAML (Model-Agnostic Meta-Learning)**: Bilevel optimization learning sensitive initializations for rapid adaptation.
3. **FOMAML & Reptile**: Scalable first-order approximations eliminating second-order Hessian computation.

**Decision Boundary (When NOT to Use)**:
- Do not use second-order MAML for very deep networks without memory checkpoints or implicit layers, as inner-loop unrolling scales memory linearly with step count $\mathcal{O}(M)$.
- Do not use Prototypical Networks when tasks require complex fine-tuning of non-linear decision boundaries rather than cluster assignment.

**Topic Bridges & Connects To**:
- `03_semi_supervised_fixmatch_mean_teacher_and_nnpu_learning` (Label efficiency under scarce supervision)
- `05_continual_learning_ewc_synaptic_intelligence_lwf_and_replay` (Sequential multi-task retention)

## 2. Mathematical Formulations & Theoretical Foundations

- **ProtoNet Centroid Prototypes**:
  $$c_k = \frac{1}{|S_k|} \sum_{(x_i, y_i) \in S_k} f_\phi(x_i), \quad \mathcal{O}(K \cdot D)$$
- **ProtoNet Query Logits (Negative Squared Euclidean Distance)**:
  $$\text{logits}_{q, k} = - \|f_\phi(x_q) - c_k\|_2^2, \quad \mathcal{O}(Q \cdot N \cdot D)$$
- **MAML Inner Linear Gradient Adaptation**:
  $$\theta'_i = \theta - \alpha \nabla_\theta \mathcal{L}_{\mathcal{T}_i}^{\text{supp}}(\theta), \quad \mathcal{O}(B_s \cdot D_{\text{in}} \cdot D_{\text{out}})$$
- **Reptile First-Order Outer Meta-Update**:
  $$\theta \leftarrow \theta + \epsilon (\theta^{(m)} - \theta), \quad \mathcal{O}(D)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "08_learning_paradigms_rl_and_alignment" / "01_self_semi_meta_continual_federated" / "04_meta_learning_prototypical_maml_and_reptile.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. Prototypical Networks Prototypes and Negative Squared Euclidean Distance
def protonet_prototypes(support_embeds: torch.Tensor) -> torch.Tensor:
    return support_embeds.mean(dim=1)

def protonet_logits(query_embeds: torch.Tensor, prototypes: torch.Tensor) -> torch.Tensor:
    return -(torch.cdist(query_embeds, prototypes, p=2) ** 2)

proto_calc = protonet_prototypes(golden["support_embeds"])
diff_proto = torch.max(torch.abs(proto_calc - golden["prototypes_ref"])).item()
print(f"ProtoNet Prototypes max abs error: {diff_proto:.2e}")
assert diff_proto < 1e-5

logits_calc = protonet_logits(golden["query_embeds"], proto_calc)
diff_logits = torch.max(torch.abs(logits_calc - golden["protonet_logits_ref"])).item()
print(f"ProtoNet Logits max abs error: {diff_logits:.2e}")
assert diff_logits < 1e-5

In [ ]:
# 2. MAML Inner Linear Gradient Step
def maml_inner_linear_step(
    x_s: torch.Tensor,
    y_s: torch.Tensor,
    w: torch.Tensor,
    b: torch.Tensor,
    alpha: float = 0.05,
) -> tuple[torch.Tensor, torch.Tensor]:
    pred_s = x_s @ w + b
    err = pred_s - y_s
    n_elements = err.numel()
    grad_w = (2.0 / n_elements) * (x_s.T @ err)
    grad_b = (2.0 / n_elements) * err.sum(dim=0)
    w_adapted = w - alpha * grad_w
    b_adapted = b - alpha * grad_b
    return w_adapted, b_adapted

w_ad, b_ad = maml_inner_linear_step(golden["x_s"], golden["y_s"], golden["w_maml"], golden["b_maml"], alpha=0.05)
diff_w = torch.max(torch.abs(w_ad - golden["w_adapted_ref"])).item()
diff_b = torch.max(torch.abs(b_ad - golden["b_adapted_ref"])).item()
print(f"MAML w_adapted error: {diff_w:.2e}, b_adapted error: {diff_b:.2e}")
assert diff_w < 1e-5 and diff_b < 1e-5

In [ ]:
# 3. Reptile Outer Meta-Update
def reptile_outer_step(
    theta: torch.Tensor,
    theta_adapted: torch.Tensor,
    epsilon: float = 0.1,
) -> torch.Tensor:
    return theta + epsilon * (theta_adapted - theta)

theta_calc = reptile_outer_step(golden["theta_base"], golden["theta_adapted"], epsilon=0.1)
diff_rep = torch.max(torch.abs(theta_calc - golden["theta_new_ref"])).item()
print(f"Reptile Meta-Update max abs error: {diff_rep:.2e}")
assert diff_rep < 1e-5

In [ ]:
# 4. Anti-Pattern / Misuse Failure Demonstration: Detached Gradient in MAML
# Demonstrating that detaching the inner-adapted weights eliminates meta-learning:
w_param = golden["w_maml"].clone().requires_grad_(True)
pred = golden["x_s"] @ w_param
loss_inner = F.mse_loss(pred, golden["y_s"])
grad = torch.autograd.grad(loss_inner, w_param, create_graph=True)[0]
w_adapted_connected = w_param - 0.05 * grad

# Correct: meta-loss wrt query set maintains second-order flow back to w_param
meta_loss_correct = F.mse_loss(golden["x_s"] @ w_adapted_connected, golden["y_s"])
meta_grad = torch.autograd.grad(meta_loss_correct, w_param)[0]
assert meta_grad.norm().item() > 0.0, "Correct MAML must produce non-zero meta-gradient"

# Anti-Pattern: Detaching severed tensor eliminates meta-gradient
w_adapted_detached = w_adapted_connected.detach().requires_grad_(True)
meta_loss_bad = F.mse_loss(golden["x_s"] @ w_adapted_detached, golden["y_s"])
grad_detached = torch.autograd.grad(meta_loss_bad, w_adapted_detached)[0]
# The gradient wrt initial parameter w_param is severed (0):
assert w_param.grad is None, "Anti-Pattern: Detached weights sever gradient propagation to initialization"

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'ProtoNet Prototypes':<32} | {diff_proto:<15.2e} | PASSED")
print(f"{'ProtoNet Negative Logits':<32} | {diff_logits:<15.2e} | PASSED")
print(f"{'MAML Linear Inner Step':<32} | {max(diff_w, diff_b):<15.2e} | PASSED")
print(f"{'Reptile Outer Meta-Update':<32} | {diff_rep:<15.2e} | PASSED")
print("=" * 65)

## 8. Summary, Key Takeaways & Cross-Domain Generalisation

- [x] **Prototypical Networks**: Compute non-parametric class centroids in metric space; squared Euclidean distance satisfies Bregman divergence cluster properties.
- [x] **MAML**: Bilevel optimization directly shapes parameter landscape so that few inner gradient steps yield optimal task adaptation.
- [x] **Reptile**: First-order meta-learning achieves competitive performance to second-order MAML without inner Hessian computation.
- [x] **Gradient Continuity**: Avoiding `.detach()` on inner adapted parameters is required for second-order meta-gradient flow.

**Cross-Domain Generalisation**:
Meta-learning transfers directly across few-shot drug activity prediction (adapting to novel molecular targets), robotic motor adaptation (adapting control policies to unexpected payload weight changes), and voice cloning (adapting acoustic models to novel speakers).

See curriculum index: [INDEX_BY_GOAL](../../../INDEX_BY_GOAL.md).